<a href="https://colab.research.google.com/github/vasnastos/StreamingAnalysis/blob/main/DL4NTP/DL4NTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL4NTP – Πρόβλεψη του μεγέθους των ροών TCP του MAWI με LSTM

Προσαρμογή του DL4NTP των Fleischer και Myerson (https://github.com/fle1scha/DL4NTP) στα αρχεία ροών του MAWI.

- Για **κάθε ημέρα χωριστά**: οι ροές TCP ταξινομούνται με τη σειρά έναρξης, το πρώτο 80% χρησιμοποιείται για εκπαίδευση και το τελευταίο 20% για έλεγχο.
- Κάθε μοντέλο βλέπει τις `N_INPUT` προηγούμενες ροές και προβλέπει τα `Bytes` της επόμενης.
- Οι ροές ελέγχου είναι οι ίδιες με εκείνες των μοντέλων αναφοράς του `analysis.py` (`Results/TCP/week_baselines.csv`), οπότε τα σφάλματα συγκρίνονται άμεσα.

**Εκτέλεση:** στο Google Colab, *Runtime → Run all*. Τοπικά: `pip install tensorflow pandas scikit-learn matplotlib jupyter` και `jupyter notebook` μέσα στον φάκελο του αποθετηρίου.

In [ ]:
import os, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, LSTM, Dense, Bidirectional
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler

SEED = 42
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## Ρυθμίσεις
Όλες οι επιλογές της πρόβλεψης βρίσκονται εδώ. Για μία μόνο ημέρα, π.χ. `DAYS = ["20260609"]`.

In [ ]:
DAYS = None              # None = και οι 7 ημέρες, ή λίστα, π.χ. ["20260609", "20260613"]
TARGET = "Bytes"         # μεταβλητή στόχος: συνολικά byte της ροής
# Υποψήφια χαρακτηριστικά, με σειρά προτεραιότητας (το Bytes πρώτο, ώστε να κρατηθεί πάντα)
CANDIDATES = ["Bytes", "Bytes B → A", "Bytes A → B", "Packets",
              "Packets A → B", "Packets B → A", "Duration"]
MAX_CORR = 0.9           # αν δύο πεδία έχουν |Spearman| > MAX_CORR, κρατιέται μόνο το πρώτο
LOG = True               # log(1+x) πριν από την κανονικοποίηση (οι κατανομές είναι πολύ ασύμμετρες)
TEST_SIZE = 0.2          # τελευταίο 20% κάθε ημέρας για έλεγχο, όπως στο analysis.py
N_INPUT = 100            # πόσες προηγούμενες ροές βλέπει το μοντέλο (όπως στο DL4NTP)
UNITS = 50               # μονάδες κάθε στρώσης LSTM (όπως στο DL4NTP)
EPOCHS = 5               # (όπως στο DL4NTP)
BATCH = 100              # (όπως στο DL4NTP)

## Δεδομένα
Αν ο φάκελος `Conversations` δεν βρεθεί (π.χ. στο Colab), το αποθετήριο κατεβαίνει αυτόματα.

In [ ]:
REPO = "https://github.com/vasnastos/StreamingAnalysis.git"
for BASE in [".", "..", "StreamingAnalysis"]:
    if os.path.isdir(os.path.join(BASE, "Conversations")):
        break
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    BASE = "StreamingAnalysis"

FILES = sorted(glob.glob(os.path.join(BASE, "Conversations", "first_5000_flows_*.csv")))
OUT_DIR = os.path.join(BASE, "Results", "TCP")
os.makedirs(OUT_DIR, exist_ok=True)
print(len(FILES), "αρχεία ροών στο", os.path.abspath(os.path.join(BASE, "Conversations")))

In [ ]:
def load_tcp(path):
    """Οι ροές TCP μιας ημέρας, με τη σειρά έναρξης (όπως ταξινομεί και το analysis.py)."""
    df = pd.read_csv(path)
    df = df[df["Protocol"] == "TCP"]
    return df.sort_values(["Rel Start", "Stream ID"], kind="mergesort").reset_index(drop=True)

days = {}
for f in FILES:
    d = os.path.basename(f)[len("first_5000_flows_"):-len(".csv")]
    if DAYS is None or d in DAYS:
        days[d] = load_tcp(f)

def label(d):
    return pd.Timestamp(d).strftime("%a %d/%m")

for d, df in days.items():
    n_test = int(len(df) * TEST_SIZE)
    print(f"{label(d)}: {len(df):,} ροές TCP  (εκπαίδευση {len(df) - n_test:,}, έλεγχος {n_test:,})")

## Επιλογή χαρακτηριστικών
Η συσχέτιση Spearman υπολογίζεται μόνο στο τμήμα εκπαίδευσης όλων των ημερών, ώστε το τμήμα ελέγχου να μην επηρεάζει την επιλογή.
Ένα πεδίο κρατιέται μόνο αν δεν συσχετίζεται ισχυρά (πάνω από `MAX_CORR`) με κάποιο πεδίο που έχει ήδη κρατηθεί.
Τα πεδία `Bits/s` δεν χρησιμοποιούνται, γιατί είναι κενά στις ροές με διάρκεια κάτω από 5 ms.

In [ ]:
train_parts = pd.concat([df.iloc[:len(df) - int(len(df) * TEST_SIZE)] for df in days.values()])
rho = train_parts[CANDIDATES].rank().corr()      # Spearman = Pearson στις τάξεις

FEATURES = []
for c in CANDIDATES:
    if all(abs(rho.loc[c, k]) <= MAX_CORR for k in FEATURES):
        FEATURES.append(c)

print(rho.round(2).to_string())
print("\nΧαρακτηριστικά:", FEATURES)

## Προετοιμασία
- `log(1+x)` και κανονικοποίηση στο [0, 1] με `MinMaxScaler`, που μαθαίνει τα όρια μόνο από το τμήμα εκπαίδευσης.
- Κάθε δείγμα είναι ένα παράθυρο με τις `N_INPUT` προηγούμενες ροές, και στόχος τα `Bytes` της επόμενης ροής.
- Τα παράθυρα του ελέγχου μπορούν να περιέχουν ροές του τμήματος εκπαίδευσης ως ιστορικό, ώστε να προβλέπεται κάθε ροή ελέγχου.

In [ ]:
def prepare(df):
    n_test = int(len(df) * TEST_SIZE)
    n_tr = len(df) - n_test
    X = df[FEATURES].to_numpy(dtype=float)
    y = df[TARGET].to_numpy(dtype=float)
    Xt, yt = (np.log1p(X), np.log1p(y)) if LOG else (X, y.copy())

    x_scaler = MinMaxScaler().fit(Xt[:n_tr])
    y_scaler = MinMaxScaler().fit(yt[:n_tr].reshape(-1, 1))
    Xs = x_scaler.transform(Xt)
    ys = y_scaler.transform(yt.reshape(-1, 1)).ravel()

    target = np.arange(N_INPUT, len(df))                       # η ροή που προβλέπεται
    W = np.stack([Xs[i - N_INPUT:i] for i in target])          # οι N_INPUT προηγούμενες ροές
    train = target < n_tr
    return {"X_train": W[train], "y_train": ys[target[train]],
            "X_test": W[~train], "y_true": y[target[~train]],
            "y": y, "n_tr": n_tr, "n_test": n_test, "y_scaler": y_scaler}

def to_bytes(p, y_scaler):
    v = y_scaler.inverse_transform(np.asarray(p).reshape(-1, 1)).ravel()
    return np.maximum(np.expm1(v) if LOG else v, 0)

In [ ]:
import textwrap

def add_caption(fig, text, fontsize=10):
    """Προσθέτει κάτω από το διάγραμμα ένα κείμενο που εξηγεί τι δείχνει."""
    w, h = fig.get_size_inches()
    wrapped = "\n\n".join(textwrap.fill(p, int(w * 13)) for p in text.split("\n\n"))
    cap_h = 0.2 * (wrapped.count("\n") + 1) + 0.35        # ύψος του κειμένου σε ίντσες
    fig.set_size_inches(w, h + cap_h)
    fig.tight_layout(rect=[0, cap_h / (h + cap_h), 1, 1])
    fig.text(0.015, (cap_h - 0.12) / (h + cap_h), wrapped, ha="left", va="top",
             fontsize=fontsize, linespacing=1.4)

## Διαχωρισμός εκπαίδευσης και ελέγχου
Όλες οι ροές TCP κάθε ημέρας, με το τμήμα της εκπαίδευσης και το τμήμα του ελέγχου σε διαφορετικό χρώμα (`week_lstm_split.png`).

In [ ]:
# Όλες οι ροές κάθε ημέρας: εκπαίδευση (πρώτο 80%) και έλεγχος (τελευταίο 20%)
fig, axes = plt.subplots(len(days), 1, figsize=(12, 2.6 * len(days)), squeeze=False)
for ax, (d, df) in zip(axes[:, 0], days.items()):
    y = df[TARGET].to_numpy(dtype=float) + 1
    n_test = int(len(df) * TEST_SIZE)
    n_tr = len(df) - n_test
    x = np.arange(len(df))
    ax.axvspan(0, N_INPUT, color="#d9d9d6", alpha=0.6, lw=0, label=f"History only (first {N_INPUT})")
    ax.plot(x[:n_tr], y[:n_tr], color="#2a78d6", lw=0.6, label=f"Training ({n_tr:,} flows)")
    ax.plot(x[n_tr:], y[n_tr:], color="#eb6834", lw=0.6, label=f"Test ({n_test:,} flows)")
    ax.axvline(n_tr, color="black", ls="--", lw=1)
    ax.set_yscale("log")
    ax.set_xlim(0, len(df))
    ax.set_title(label(d), loc="left")
    ax.set_ylabel("Bytes + 1")
    ax.legend(loc="upper left", ncol=3, fontsize=8, framealpha=0.9)
axes[-1, 0].set_xlabel("Flow (in order of start time)")

add_caption(fig,
    f"What the figure shows: one panel per day with all the TCP flows of that day. The horizontal axis is the "
    f"flow number, in order of start time, and the vertical axis is the size of each flow in bytes (+1, "
    f"logarithmic scale: each grid line is ten times the previous one). The blue part is the training set, "
    f"the first {1 - TEST_SIZE:.0%} of the flows, from which the models learn. The orange part is the test set, "
    f"the last {TEST_SIZE:.0%} of the flows, which the models do not see during training and on which they are "
    f"evaluated. The dashed line marks the split. Each prediction needs the {N_INPUT} previous flows, so the "
    f"first {N_INPUT} flows of each day (grey band) are used only as history and are not predicted themselves.")
fig.savefig(os.path.join(OUT_DIR, "week_lstm_split.png"), dpi=150)
plt.show()

## Μοντέλα
Οι τρεις αρχιτεκτονικές του DL4NTP, με τις ίδιες παραμέτρους.

In [ ]:
def rmse(y_true, y_pred):
    # Όπως στο DL4NTP. Με μία έξοδο, το axis=-1 δίνει |σφάλμα| ανά δείγμα,
    # οπότε στην πράξη η εκπαίδευση ελαχιστοποιεί το μέσο απόλυτο σφάλμα.
    return tf.sqrt(tf.reduce_mean(tf.square(y_pred - y_true), axis=-1))

def simple_lstm(n_features):
    m = Sequential([Input((N_INPUT, n_features)),
                    LSTM(UNITS, activation="sigmoid"),
                    Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    return m

def bidirectional_lstm(n_features):
    m = Sequential([Input((N_INPUT, n_features)),
                    Bidirectional(LSTM(UNITS, activation="sigmoid")),
                    Dense(1)])
    m.compile(optimizer="adam", loss=rmse)
    return m

def stacked_lstm(n_features):
    m = Sequential([Input((N_INPUT, n_features)),
                    LSTM(UNITS, return_sequences=True, activation="sigmoid"),
                    LSTM(UNITS, return_sequences=True),
                    LSTM(UNITS),
                    Dense(1)])
    m.compile(optimizer="adam", loss=rmse)
    return m

MODELS = {"Simple LSTM": simple_lstm,
          "Bidirectional LSTM": bidirectional_lstm,
          "Stacked LSTM": stacked_lstm}

## Εκπαίδευση και έλεγχος ανά ημέρα
Για κάθε ημέρα εκπαιδεύονται τα τρία μοντέλα από την αρχή. Στις ίδιες ροές ελέγχου υπολογίζονται και τρία μοντέλα αναφοράς του `analysis.py`.
Το R² υπολογίζεται όπως στο DL4NTP, ως το τετράγωνο της συσχέτισης πρόβλεψης και πραγματικής τιμής.

In [ ]:
def metrics(y_true, y_pred):
    r2 = np.corrcoef(y_true, y_pred)[0, 1] ** 2 if np.std(y_pred) > 0 else np.nan
    return {"MAE": np.mean(np.abs(y_true - y_pred)),
            "RMSE": np.sqrt(np.mean((y_true - y_pred) ** 2)),
            "R2": r2}

rows, predictions, histories = [], {}, {}
for d, df in days.items():
    p = prepare(df)
    y, n_tr, n_test = p["y"], p["n_tr"], p["n_test"]
    s = pd.Series(y)
    pred = {"Previous flow": y[n_tr - 1:-1],
            "Median of previous 100 flows": s.rolling(100).median().shift(1).to_numpy()[n_tr:],
            "Train median": np.full(n_test, np.median(y[:n_tr]))}

    for name, build in MODELS.items():
        keras.utils.set_random_seed(SEED)
        model = build(len(FEATURES))
        h = model.fit(p["X_train"], p["y_train"], epochs=EPOCHS, batch_size=BATCH,
                      shuffle=False, verbose=0)
        pred[name] = to_bytes(model.predict(p["X_test"], verbose=0), p["y_scaler"])
        histories[(label(d), name)] = h.history["loss"]
        print(f"{label(d)}  {name:<19} τελικό loss {h.history['loss'][-1]:.4f}")

    for name, pr in pred.items():
        rows.append({"Day": label(d), "Model": name, "Test flows": n_test, **metrics(p["y_true"], pr)})
    predictions[label(d)] = (p["y_true"], pred)

results = pd.DataFrame(rows)
results.to_csv(os.path.join(OUT_DIR, "week_lstm.csv"), index=False)
print("\n[SAVED]", os.path.join(OUT_DIR, "week_lstm.csv"))

## Αποτελέσματα
Μέσο απόλυτο σφάλμα (MAE) σε byte ανά ημέρα και μοντέλο. Μικρότερη τιμή σημαίνει καλύτερη πρόβλεψη.

In [ ]:
order = list(pred.keys())
for metric in ["MAE", "RMSE", "R2"]:
    t = results.pivot(index="Model", columns="Day", values=metric).loc[order, [label(d) for d in days]]
    t["Mean"] = t.mean(axis=1)
    print(f"\n{metric}")
    print(t.to_string(float_format=lambda v: f"{v:,.3f}" if metric == "R2" else f"{v:,.0f}"))

In [ ]:
# Πραγματικά (Actual) και προβλεπόμενα byte στις ροές ελέγχου κάθε ημέρας (λογαριθμική κλίμακα)
COLORS = {"Simple LSTM": "C0", "Bidirectional LSTM": "C1", "Stacked LSTM": "C2"}
mae = results.pivot(index="Model", columns="Day", values="MAE")
actual = np.concatenate([t for t, _ in predictions.values()])

def plot_predictions(models, filename, caption):
    fig, axes = plt.subplots(len(predictions), 1, figsize=(12, 3 * len(predictions)), squeeze=False)
    for ax, (lab, (y_true, pred)) in zip(axes[:, 0], predictions.items()):
        ax.plot(y_true + 1, color="#a3a29c", lw=0.8, label="Actual")
        for name in models:
            ax.plot(pred[name] + 1, lw=1, color=COLORS[name], label=name)
        ax.set_yscale("log")
        title = lab
        if len(models) == 1:
            title += (f"   MAE {mae.loc[models[0], lab]:,.0f} bytes"
                      f"  (always predicting the training median: {mae.loc['Train median', lab]:,.0f})")
        ax.set_title(title, loc="left")
        ax.set_ylabel("Bytes + 1")
    axes[0, 0].legend(ncol=4, fontsize=8)
    axes[-1, 0].set_xlabel("Test flow")
    add_caption(fig, caption)
    fig.savefig(os.path.join(OUT_DIR, filename), dpi=150)
    plt.show()

# 1) Και τα τρία μοντέλα μαζί
lstm = np.concatenate([p[m] for _, p in predictions.values() for m in MODELS])
plot_predictions(list(MODELS), "week_lstm_predictions.png",
    f"What the figure shows: one panel per day. The horizontal axis lists the TCP flows of the test set, "
    f"i.e. the last {TEST_SIZE:.0%} of the day's flows in order of start time, which the models did not see "
    f"during training. The vertical axis is the size of each flow in bytes (+1, logarithmic scale: each "
    f"grid line is ten times the previous one). The grey line is the actual size of each flow. The coloured "
    f"lines are the sizes predicted by the three LSTM models for the same flow, each prediction made only "
    f"from the {N_INPUT} flows that came before it. Where a coloured line meets the grey line, the "
    f"prediction is correct."
    f"\n\n"
    f"In these results the actual flows range from {actual.min():,.0f} to {actual.max():,.0f} bytes "
    f"(median {np.median(actual):,.0f}), while the LSTM predictions range only from {lstm.min():,.0f} "
    f"to {lstm.max():,.0f} bytes (median {np.median(lstm):,.0f}).")

# 2) Κάθε μοντέλο χωριστά
for name in MODELS:
    own = np.concatenate([p[name] for _, p in predictions.values()])
    slug = name.split()[0].lower()
    plot_predictions([name], f"week_lstm_predictions_{slug}.png",
        f"What the figure shows: the predictions of {name} alone, one panel per day. The horizontal axis lists "
        f"the TCP flows of the test set, i.e. the last {TEST_SIZE:.0%} of the day's flows in order of start time, "
        f"which the model did not see during training. The vertical axis is the size of each flow in bytes (+1, "
        f"logarithmic scale: each grid line is ten times the previous one). The grey line is the actual size of "
        f"each flow and the coloured line is the size predicted by {name} for the same flow, using only the "
        f"{N_INPUT} flows that came before it. The title of each panel gives the model's mean absolute error "
        f"(MAE) for that day and, for comparison, the MAE of always predicting the median of the training flows."
        f"\n\n"
        f"Across the {len(predictions)} days the predictions of {name} range from {own.min():,.0f} to "
        f"{own.max():,.0f} bytes (median {np.median(own):,.0f}), while the actual flows range from "
        f"{actual.min():,.0f} to {actual.max():,.0f} bytes (median {np.median(actual):,.0f}). Its mean MAE is "
        f"{mae.loc[name].mean():,.0f} bytes, against {mae.loc['Train median'].mean():,.0f} bytes for the "
        f"training median.")

In [ ]:
# Loss ανά εποχή
fig, ax = plt.subplots(1, len(MODELS), figsize=(14, 3.5), squeeze=False)
for a, name in zip(ax[0], MODELS):
    for (lab, n), loss in histories.items():
        if n == name:
            a.plot(range(1, len(loss) + 1), loss, marker="o", label=lab)
    a.set_title(name); a.set_xlabel("Epoch"); a.set_ylabel("Loss")
    a.set_xticks(range(1, EPOCHS + 1))
ax[0, 0].legend(fontsize=7)

n_train = [len(df) - int(len(df) * TEST_SIZE) - N_INPUT for df in days.values()]
add_caption(fig,
    f"What the figure shows: the training error (loss) of each model after every epoch, with one line per "
    f"day. An epoch is one full pass over the training data of that day: {min(n_train):,}–{max(n_train):,} "
    f"windows of {N_INPUT} flows, fed to the model in batches of {BATCH}. A falling line means that the "
    f"model is still learning; a flat line means that further epochs bring little improvement. "
    f"The loss is measured on the normalised log(1 + bytes) scale, not in bytes. Simple LSTM is trained with "
    f"the mean squared error, while Bidirectional and Stacked LSTM use the rmse function of DL4NTP, which with "
    f"a single output equals the mean absolute error, so the values are not comparable between the three panels.")
fig.savefig(os.path.join(OUT_DIR, "week_lstm_loss.png"), dpi=150)
plt.show()

## Λήψη των αποτελεσμάτων
Όλα τα αρχεία (CSV και PNG) μπαίνουν σε ένα zip, `week_lstm_results.zip`, που στο Colab κατεβαίνει με μία λήψη.
Αν η λήψη δεν ξεκινήσει, το zip βρίσκεται στο αριστερό μενού του Colab (εικονίδιο φακέλου) → `StreamingAnalysis/Results/TCP` → δεξί κλικ → *Download*.

In [ ]:
import zipfile

OUT_FILES = ["week_lstm.csv", "week_lstm_split.png", "week_lstm_predictions.png",
             "week_lstm_predictions_simple.png", "week_lstm_predictions_bidirectional.png",
             "week_lstm_predictions_stacked.png", "week_lstm_loss.png"]
zip_path = os.path.join(OUT_DIR, "week_lstm_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT_FILES:
        path = os.path.join(OUT_DIR, f)
        if os.path.exists(path):
            z.write(path, arcname=f)
            print("[ZIP]", f)
        else:
            print("[MISSING]", f)
print("\n[SAVED]", os.path.abspath(zip_path))

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass